TRANSFER LEARNING AND FINE TUNNING WITH BERT

In [2]:
from transformers import BertTokenizer
from datasets import load_dataset,ClassLabel
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
from transformers import TrainingArguments, Trainer, BertForSequenceClassification
from transformers import BertModel

W0525 04:23:13.166000 11548 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [3]:

dataset=load_dataset('emotion',split='train')

In [4]:
dataset=dataset.filter(lambda x:x['label'] in [0,1,3])
label_map={0: 0, 1: 1, 3: 2}
dataset=dataset.map(lambda x: {'label': label_map[x['label']]})
new_label_features=ClassLabel(num_classes=3,names=['sadness','joy','anger'])
dataset=dataset.cast_column('label',new_label_features)
print(dataset.features["label"].names)

['sadness', 'joy', 'anger']


In [5]:
dataset=dataset.train_test_split(test_size=0.2)

In [6]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 9749
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2438
    })
})

In [7]:
train_texts=dataset['train']['text']

In [8]:
train_texts

Column(['i am feeling disheartened with my words as of late', 'i feel empty and dim if i miss that', 'i miss not feeling guilt over so much stuff because i reacted in a terrible way or said no to my kids just for the sake of saying no', 'i tend to feel a bit cranky when i ve gone for a few days without making art', 'i m feeling miserable serioulsy'])

In [9]:
train_labels=dataset['train']['label']
train_labels
print(len(train_texts),len(train_labels))

9749 9749


In [10]:
tokenizer=BertTokenizer.from_pretrained('bert-base-uncased')

In [11]:
def tokenize(batch):
    return tokenizer(batch['text'],padding=True,truncation=True,return_tensors='pt')

In [12]:
dataset=dataset.map(lambda x: tokenizer(x['text'],padding='max_length',truncation=True),batched=True)

Map:   0%|          | 0/9749 [00:00<?, ? examples/s]

Map:   0%|          | 0/2438 [00:00<?, ? examples/s]

In [13]:
dataset.set_format(type='torch',columns=['input_ids','attention_mask','label'])

In [14]:
model=BertForSequenceClassification.from_pretrained('bert-base-uncased')

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [15]:
for name, param in model.named_parameters():
    print(name,param.requires_grad)

bert.embeddings.word_embeddings.weight True
bert.embeddings.position_embeddings.weight True
bert.embeddings.token_type_embeddings.weight True
bert.embeddings.LayerNorm.weight True
bert.embeddings.LayerNorm.bias True
bert.encoder.layer.0.attention.self.query.weight True
bert.encoder.layer.0.attention.self.query.bias True
bert.encoder.layer.0.attention.self.key.weight True
bert.encoder.layer.0.attention.self.key.bias True
bert.encoder.layer.0.attention.self.value.weight True
bert.encoder.layer.0.attention.self.value.bias True
bert.encoder.layer.0.attention.output.dense.weight True
bert.encoder.layer.0.attention.output.dense.bias True
bert.encoder.layer.0.attention.output.LayerNorm.weight True
bert.encoder.layer.0.attention.output.LayerNorm.bias True
bert.encoder.layer.0.intermediate.dense.weight True
bert.encoder.layer.0.intermediate.dense.bias True
bert.encoder.layer.0.output.dense.weight True
bert.encoder.layer.0.output.dense.bias True
bert.encoder.layer.0.output.LayerNorm.weight True


In [16]:
model_for_cls=BertForSequenceClassification.from_pretrained('bert-base-uncased',num_labels=3)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [17]:
for name, param in model.named_parameters():
    print(name,param.requires_grad)

bert.embeddings.word_embeddings.weight True
bert.embeddings.position_embeddings.weight True
bert.embeddings.token_type_embeddings.weight True
bert.embeddings.LayerNorm.weight True
bert.embeddings.LayerNorm.bias True
bert.encoder.layer.0.attention.self.query.weight True
bert.encoder.layer.0.attention.self.query.bias True
bert.encoder.layer.0.attention.self.key.weight True
bert.encoder.layer.0.attention.self.key.bias True
bert.encoder.layer.0.attention.self.value.weight True
bert.encoder.layer.0.attention.self.value.bias True
bert.encoder.layer.0.attention.output.dense.weight True
bert.encoder.layer.0.attention.output.dense.bias True
bert.encoder.layer.0.attention.output.LayerNorm.weight True
bert.encoder.layer.0.attention.output.LayerNorm.bias True
bert.encoder.layer.0.intermediate.dense.weight True
bert.encoder.layer.0.intermediate.dense.bias True
bert.encoder.layer.0.output.dense.weight True
bert.encoder.layer.0.output.dense.bias True
bert.encoder.layer.0.output.LayerNorm.weight True


In [18]:
total_params=sum(p.numel() for p in model.parameters())
print(total_params)

109483778


In [19]:
trainable_params=sum(p.numel() for p in model.parameters() if p.requires_grad)
print(trainable_params )

109483778


In [20]:
print(model_for_cls.bert.embeddings)

BertEmbeddings(
  (word_embeddings): Embedding(30522, 768, padding_idx=0)
  (position_embeddings): Embedding(512, 768)
  (token_type_embeddings): Embedding(2, 768)
  (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
  (dropout): Dropout(p=0.1, inplace=False)
)


In [21]:
print(model_for_cls.bert.encoder.layer[11])

BertLayer(
  (attention): BertAttention(
    (self): BertSdpaSelfAttention(
      (query): Linear(in_features=768, out_features=768, bias=True)
      (key): Linear(in_features=768, out_features=768, bias=True)
      (value): Linear(in_features=768, out_features=768, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (output): BertSelfOutput(
      (dense): Linear(in_features=768, out_features=768, bias=True)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (intermediate): BertIntermediate(
    (dense): Linear(in_features=768, out_features=3072, bias=True)
    (intermediate_act_fn): GELUActivation()
  )
  (output): BertOutput(
    (dense): Linear(in_features=3072, out_features=768, bias=True)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
)


In [22]:
print(model_for_cls.classifier)

Linear(in_features=768, out_features=3, bias=True)


In [23]:
print(model_for_cls.config)

BertConfig {
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "dtype": "float32",
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "LABEL_2": 2
  },
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.57.1",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}



In [24]:
training_args = TrainingArguments(
    output_dir='./results',     
    save_strategy='epoch',       
    per_device_train_batch_size=8,  
    per_device_eval_batch_size=8,    
    num_train_epochs=1,         
    logging_dir='./logs',        
    report_to='none',            
)

In [25]:
import os
os.environ['WANDB_DISABLED']='true'

In [1]:
trainer=Trainer(
    model=model_for_cls,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test']
)

NameError: name 'Trainer' is not defined

In [ ]:
trainer.train()

c:\Users\QHS 006\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


UNFREEZE COUPLE OF LAST LAYER AND THEN TRAIN IT

In [ ]:
from transformers import BertModel,BertTokenizer,PreTrainedModel,Trainer,TrainingArguments,BertConfig
from transformers.modeling_outputs import SequenceClassifierOutput

In [ ]:
bert=BertModel.from_pretrained('bert-base-uncased')

In [ ]:
for param in bert.parameters():
    param.requires_grad = False

In [ ]:
bert.encoder

In [ ]:
bert.encoder.layer

In [ ]:
for layer in bert.encoder.layer:
    print(layer)

In [ ]:
for layer in bert.encoder.layer[-2:]:
    for param in layer.parameters():
        param.requires_grad = True

In [ ]:
class BertMulticlassClassifier(PreTrainedModel):
    def __init__(self,config):
        super().__init__(config)
        self.bert=BertModel.from_pretrained('bert-base-uncased',config=config)
        self.dropout=nn.Dropout(0.3)
        self.classifier=nn.Linear(config.hidden_size,config.num_labels)

        for param in self.bert.parameters():
            param.requires_grad=False

        for layer in self.bert.encoder.layer[-2:]:
            for param in layer.parameters():
                param.requires_grad=True

        for param in self.bert.pooler.parameters():
            param.requires_grad=True

    def forward(self,input_ids=None,attention_mask=None,labels=None):
        outputs=self.bert(input_ids=input_ids,attention_mask=attention_mask)
        pooled_output=outputs.pooler_output
        logits=self.classifier(pooled_output)

        loss=None
        if labels is not None:
            loss_fct=nn.CrossEntropyLoss()
            loss=loss_fct(logits.view(-1,self.config.num_labels),labels.view(-1))

        return SequenceClassifierOutput(loss=loss,logits=logits)

In [ ]:
config=BertConfig.from_pretrained('bert-base-uncased',num_labels=3)

In [ ]:
model=BertMulticlassClassifier(config)

In [ ]:
training_Args=TrainingArguments(
    output_dir='./results',
    save_strategy='epoch',
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    logging_dir='./logs',
    report_to='none',
)

In [ ]:
trainer=Trainer(
    model=model_for_cls,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test']
)

In [ ]:
trainer.train()

In [ ]:
label_map={0: 'sadness', 1: 'joy', 2: 'anger'}
def predict(text,model,tokenizer,label_map):
    model.eval()
    inputs=tokenizer(text,padding='max_length',truncation=True,return_tensors='pt')
    with torch.no_grad():
        outputs=model(**inputs)
        logits=outputs.logits
        predicted_label_id=torch.argmax(logits,dim=1).item()
        predicted_label=label_map[predicted_label_id]
    return predicted_label

In [ ]:
predict("I am feeling very happy today!",model,label_map)
predict("I am feeling very sad today!",model,label_map)